### Analyse the clusters found by GenDSI and name the clusters by the most frequent slot name

In [26]:
import torch
from collections import Counter, defaultdict
from typing import List, Dict, Any
import random
import pprint

In [34]:
dataset = "multiwoz21"

In [35]:
slotcluster_list = torch.load(f"clusters/{dataset}_clusters.pt")

/var/folders/vn/5z6n4f1s5zdgyyc7wh6kqn4m0000gn/T/ipykernel_6570/531013404.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  slotcluster_list = torch.load(f"clusters/{datas

In [36]:
#for each cluster find the most common slot name and make a dict where the values are assigned
def extract_most_common_slot_mapping(data: List[List[str]]) -> Dict[str, List[str]]:
    """
    Processes a list of lists containing key-value string pairs separated by a colon (":").
    For each sublist, identifies the most frequent slot name (key) and aggregates all values 
    under that slot name. Returns a single dictionary mapping each sublist's most common slot
    to its collected values.

    Args:
        data (List[List[str]]): A list of lists, where each inner list contains strings
                                formatted as "slotname: value".

    Returns:
        Dict[str, List[str]]: A dictionary with the most common slot name from each sublist
                              as key and a list of all values as its value.
    """
    result = {}

    for sublist in data:
        slot_values = defaultdict(list)
        slots = []

        for entry in sublist:
            if ':' in entry:
                slot, value = map(str.strip, entry.split(':', 1))
                slots.append(slot)
                slot_values[slot].append(value)

        if not slots:
            continue  # Skip if no valid slot-value pairs

        most_common_slot = Counter(slots).most_common(1)[0][0]
        all_values = [v for values in slot_values.values() for v in values]
        result[most_common_slot] = list(set(all_values))

    return result


In [37]:
slotvalue_dict = extract_most_common_slot_mapping(slotcluster_list)

In [38]:
#print the slotnames and the number of values plus some random sample of the values
for slot, values in slotvalue_dict.items():
    print(f"slot {slot} with {len(values)} values, such as: {random.sample(values, min(10, len(values)))}")
    print()
    

slot aldous harding location with 20 values, such as: ['Lodge Room', 'The Lodge Room', '6 pm', 'Booking 1 ticket to the Aldous Harding Event on March 6th in Los Angeles.', 'March 13th.', 'Aldous Harding at the Lodge Room on march 5th at 5 pm.', 'Aldous Harding at Lodge Room', '5 pm', '4:30 pm.', 'Aldous Harding Event.']

slot event with 15 values, such as: ['Advanced acting scene study at tgw acting studio on march 7th at 6:30 pm', 'advanced acting scene study at tgw acting studio on march 7th at 6:30 pm', 'tgw acting studio', 'Advanced Acting Scene study', 'TGW Acting studio', 'TGW Acting Studio.', '6:30 pm', 'TGW Acting Studio', 'Dearing acting studio', 'Advanced Acting Scene Study at TGW Acting Studio on March 10th at 4:30 pm.']

slot show name with 9 values, such as: ['The theatre raymond kabbaz.', '7 pm', 'Adieu Monsieur Haffmann', 'Adieu monsieur haffmann.', 'march 12th', 'theatre raymond kabbaz', 'Adieu monsieur haffmann at the theatre raymond kabbaz on march 4th at 6:30 pm.', '

### now add the domains if possible from the most frequent slot names

In [39]:
def build_domain_slot_value_hierarchy(flat_slot_dict: Dict[str, List[str]]) -> Dict[str, Dict[str, List[str]]]:
    """
    Builds a domain-slot-value hierarchy from a flat slot-value dictionary.

    - If a slot name has multiple words, the first word is used as the domain,
      and the rest are joined as the new slot name.
    - If a slot name has only one word, the domain is 'general' and the slot remains unchanged.

    Args:
        flat_slot_dict (Dict[str, List[str]]): Dictionary with original slot names as keys 
                                               and lists of values as values.

    Returns:
        Dict[str, Dict[str, List[str]]]: Nested dictionary with domains as top-level keys,
                                         and slot-value mappings inside each domain.
    """
    hierarchy = defaultdict(dict)

    for original_slot, values in flat_slot_dict.items():
        words = original_slot.strip().split()

        if len(words) > 1:
            domain = words[0]
            slot = ' '.join(words[1:])
        else:
            domain = 'general'
            slot = words[0]

        hierarchy[domain][slot] = values

    return dict(hierarchy)

In [40]:
domain_slotvalue_dict = build_domain_slot_value_hierarchy(slotvalue_dict)

In [41]:
pprint.pprint(domain_slotvalue_dict)

{'30': {'St Mary Axe': ['10 attractions, 30 St Mary Axe (The Gherkin) is a '
                        'Historical Landmark',
                        '10 attractions, 30 St Mary Axe (The Gherkin)',
                        'The Gherkin: Historical Landmark',
                        'Historical landmark',
                        '30 St Mary Axe (the Gherkin)',
                        '30 St Mary Axe (The Gherkin)',
                        'Historical Landmark at 30 St Mary Axe (The Gherkin)',
                        'True',
                        '30 St mary Axe (The Gherkin)',
                        'Heathrow International Airport, 30 St Mary Axe (The '
                        'Gherkin) or some other address',
                        'The Gherkin',
                        'Historical Landmark',
                        'historical Landmark',
                        '30 st Mary axe (the gherkin)',
                        '30 St Mary Axe (The Gherkin), Historical Landmark',
               

In [33]:
#save the dictionary to evaluate it later on
torch.save(domain_slotvalue_dict, f"ontology_hierarchy_dicts/{dataset}_genDSI_ontology_prediction.pt")